# HarrisWGAN demo

In [ ]:
import os, glob
import sys
from typing import List, Tuple, Union, Dict
from pathlib import Path
from abc import ABC, abstractmethod
from datetime import datetime as dt
from pathlib import Path
import re
from operator import itemgetter
from functools import partial
import importlib
import gc
import json as js
from timeit import default_timer as timer
import random

import numpy as np
import xarray as xr
import dask
import tensorflow as tf

import multiprocessing
try:
    from multiprocessing import Pool as ThreadPool
except:
    from multiprocessing.pool import ThreadPool

repo_dir = Path("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/")
sys.path.append(str(repo_dir.joinpath("models")))
sys.path.append(str(repo_dir.joinpath("utils")))
sys.path.append(str(repo_dir.joinpath("handle_data")))
from model_engine import ModelEngine
from other_utils import find_closest_divisor

### Data pipeline for input streams of Harris WGAN 

The input differs from the other baseline models in a way that the low-res input data is not upscaled (bi-linearly interpolated) to the target grid and that high-res static data constitues another input stream. Thus, the generator yields a dictionary which is then used to set-upthe TF data pipeline.
To-Do: 
 - [ ] implement updated make_tf_dataset_allmem-method in handle_data_class.py
 - [ ] adapt make_tf_dataset_dyn-method accordingly

In [ ]:
def split_in_tar(
    ds: xr.Dataset,
    predictands: List = None,
    predictors: List = None,
    static_vars: List = None,
) -> Tuple[xr.Dataset, xr.Dataset]:
    """
    Split data array with variables-dimension into input and target data for downscaling
    :param da: The unsplitted data array
    :param target_var: Name of target variable which should consttute the first channel
    :param predictands: List of selected predictand variables; parse None to use
                        all predictands (vars with suffix _tar)
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :return: The split data array.
    """
    varnames = list(ds.data_vars)

    if predictors is None:
        invars = [var for var in varnames if var.endswith("_in")]
    else:
        assert all(
            [predictor in varnames for predictor in predictors]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        invars = list(predictors)
    if predictands is None:
        tarvars = [var for var in varnames if var.endswith("_tar")]
    else:
        assert all(
            [predictand in varnames for predictand in predictands]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        tarvars = list(predictands)

    if static_vars is None:
        ds_in, ds_tar = ds[invars], ds[tarvars]

        return ds_in, ds_tar
    else:
        assert all(
            [static_var in varnames for static_var in static_vars]
        ), f"At least ostatic high-res is not a data variable. Available variables are {*varnames,}"
        statvars = list(static_vars)

        ds_in, ds_tar, ds_stat = ds[invars], ds[tarvars], ds[statvars]

        return ds_in, ds_tar, ds_stat


def reshape_ds(ds):
    """
    Convert a xarray dataset to a data-array where the variables will constitute the last dimension (channel last)
    :param ds: the xarray dataset with dimensions (dims)
    :return da: the data-array with dimensions (dims, variables)
    """
    da = ds.to_array(dim="variables")
    da = da.transpose(..., "variables")
    return da


def make_tf_dataset_allmem(
    ds: xr.Dataset,
    batch_size: int,
    predictands: List,
    predictors: List,
    static_vars: List,
    lshuffle: bool = True,
    shuffle_samples: int = 20000,
    named_targets: bool = False,
    var_tar2in: str = None,
    lrepeat: bool = True,
    drop_remainder: bool = True,
) -> tf.data.Dataset:
    """
    Build-up TensorFlow dataset from a generator based on the xarray-data array.
    NOTE: All data is loaded into memory
    :param ds: the xarray dataset. Input variable names must carry the suffix '_in', whereas it must be '_tar' for target variables
    :param batch_size: number of samples per mini-batch
    :param predictands: List of selected predictand variables
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :param lshuffle: flag if shuffling should be applied to dataset
    :param shuffle_samples: number of samples to load before applying shuffling
    :param named_targets: flag if target of TF dataset should be dictionary with named target variables
    :param var_tar2in: name of target variable to be added to input (used e.g. for adding high-resolved topography
                                                                        to the input)
    :param lrepeat: flag if dataset should be repeated
    :param drop_remainder: flag if samples will be dropped in case batch size is not a divisor of # data samples
    :param with_horovod: flag to trigger horovod-based distributed dataset creation
    :param lembed: flag to trigger temporal embedding (not implemented yet!)
    """

    # add time dimension to constant variables
    for var in ds.data_vars:
        if "time" not in ds[var].dims:
            ds[var] = ds[var].expand_dims({"time": ds["time"]}, axis=0)

    ds_in, ds_tar, ds_stat = split_in_tar(
        ds, predictands=predictands, predictors=predictors, static_vars=static_vars
    )

    # convert dataset to data arrays and load into memory
    da_in, da_tar, da_stat = (
        reshape_ds(ds_in).astype("float32", copy=True),
        reshape_ds(ds_tar).astype("float32", copy=True),
        reshape_ds(ds_stat).astype("float32", copy=True),
    )

    if var_tar2in is not None:
        # NOTE: * The order of the following operation must be the same as in StreamMonthlyNetCDF.getitems
        #       * The following operation order must concatenate var_tar2in by da_in to ensure
        #         that the variable appears at first place. This is required to avoid
        #         that var_tar2in becomes a predeictand when slicing takes place in tf_split
        da_in = xr.concat([da_tar.sel({"variables": var_tar2in}), da_in], "variables")

    varnames_tar = da_tar["variables"].values

    def gen_named(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            tar_now = darr_tar.isel({"time": t})
            yield tuple(
                (
                    darr_in.isel({"time": t}).values,
                    {
                        var: tar_now.sel({"variables": var}).values
                        for var in varnames_tar
                    },
                )
            )

    def gen_unnamed(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (darr_in.isel({"time": t}).values, darr_tar.isel({"time": t}).values)
            )

    def gen_dict(darr_in, darr_tar, darr_stat):
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (
                    {
                        "lo_res_inputs": darr_in.isel({"time": t}).values,
                        "hi_res_inputs": darr_stat.isel({"time": t}).values,
                    },
                    {"output": darr_tar.isel({"time": t}).values},
                )
            )

    if named_targets is True:
        gen_now = gen_named
    elif static_vars is not None:
        gen_now = gen_dict
    else:
        gen_now = gen_unnamed

    # create output signatures from first sample
    if static_vars is None:
        s0 = next(iter(gen_now(da_in, da_tar)))
        sample_spec_in = tf.TensorSpec(s0[0].shape, dtype=s0[0].dtype)
        if named_targets is True:
            sample_spec_tar = {
                var: tf.TensorSpec(s0[1][var].shape, dtype=s0[1][var].dtype)
                for var in varnames_tar
            }
        else:
            sample_spec_tar = tf.TensorSpec(s0[1].shape, dtype=s0[1].dtype)

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar)

    else:
        s0 = next(iter(gen_now(da_in, da_tar, da_stat)))

        sample_spec_in = {
            "lo_res_inputs": tf.TensorSpec(
                s0[0]["lo_res_inputs"].shape, dtype=s0[0]["lo_res_inputs"].dtype
            ),
            "hi_res_inputs": tf.TensorSpec(
                s0[0]["hi_res_inputs"].shape, dtype=s0[0]["hi_res_inputs"].dtype
            ),
        }

        sample_spec_tar = {
            "output": tf.TensorSpec(s0[1]["output"].shape, dtype=s0[1]["output"].dtype)
        }

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar, da_stat)

    data_iter = tf.data.Dataset.from_generator(
        lambda: gen_train, output_signature=(sample_spec_in, sample_spec_tar)
    )

    # Notes:
    # * cache is reuqired to make repeat work properly on datasets based on generators
    #   (see https://stackoverflow.com/questions/60226022/tf-data-generator-keras-repeat-does-not-work-why)
    # * repeat must be applied after shuffle to get varying mini-batches per epoch
    # * batch-size is increased to allow substepping in train_step
    if lshuffle > 1:
        data_iter = (
            data_iter.cache()
            .shuffle(shuffle_samples)
            .batch(batch_size, drop_remainder=drop_remainder)
        )
    else:
        data_iter = data_iter.cache().batch(batch_size, drop_remainder=drop_remainder)

    if lrepeat:
        data_iter = data_iter.repeat()

    # clean-up to free some memory
    # free_mem([da, da_in, da_tar, varnames_tar])
    del ds
    del ds_in
    del ds_tar
    del da_in
    del da_tar
    gc.collect()

    return data_iter

In [ ]:
def make_tf_dataset_dyn(ds_obj, batch_size: int, nepochs: int, nshuffle: int, lrepeat: bool = True, drop_remainder: bool = True) -> tf.data.Dataset:
    """
    Build TensorFlow dataset by streaming from netCDF using xarray's open_mfdatset-method.
    To fit into memory, only a subset of all netCDF-files is processed at once (nfiles2merge-parameter).
    :param ds_obj: StreamMonthlyNetCDF-object
    :param batch_size: desired mini-batch size
    :param nepochs: (effective) number of epochs for training
    :param nshuffle: number of samples to shuffle (set to 1 to disable shuffling)
    :param lrepeat: flag if dataset should be repeated
    :param drop_remainder: flag if samples will be dropped in case batch size is not a divisor of # data samples
    :return: TensorFlow dataset object that streams data from subset of many netCDF-files
    """
    tf_read_nc = lambda ind_set: tf.py_function(ds_obj.read_netcdf, [ind_set], tf.int64)
    tf_choose_data = lambda il: tf.py_function(ds_obj.choose_data, [il], tf.bool)
    tf_getdata = lambda i: tf.numpy_function(ds_obj.getitems, [i], tf.float32)
    
    mode = ds_obj.stream_mode
    
    if mode in ["hi_input", "hi_input_named_target"]:
        tf_getdata = lambda i: tf.numpy_function(ds_obj.getitems, [i], tf.float32)
        if mode == "hi_input":
            tf_split = lambda arr: (arr[..., 0:-ds_obj.n_predictands], arr[..., -ds_obj.n_predictands:])
        else:
            varnames = ds_obj.predictand_list
            tf_split = lambda arr: (arr[..., 0:-ds_obj.n_predictands],
                                    {var: arr[..., -ds_obj.n_predictands + i] for i, var in enumerate(varnames)})
    else: 
        def make_dict(darr_in, darr_stat, darr_out):
            return ({"lo_res_inputs": darr_in, "hi_res_inputs": darr_stat}, {"output": darr_out})
                                         
        tf_getdata = lambda i: tf.numpy_function(ds_obj.getitems, [i], [tf.float32, tf.float32, tf.float32])
        tf_split = lambda arr_in, arr_stat, arr_out: make_dict(arr_in, arr_stat, arr_out)

    # enable flexibility in factor for range
    n_reads = int(ds_obj.nfiles_merged*nepochs)
    if ds_obj.with_horovod:
        import horovod.tensorflow as hvd
        tfds = tf.data.Dataset.range(n_reads).shard(hvd.size(), hvd.rank()).map(tf_read_nc).prefetch(1)
    else:
        tfds = tf.data.Dataset.range(n_reads).map(tf_read_nc).prefetch(1)

    tfds = tfds.flat_map(lambda x: tf.data.Dataset.from_tensors(x).map(tf_choose_data))
    tfds = tfds.flat_map(
        lambda x: tf.data.Dataset.range(ds_obj.samples_merged).shuffle(nshuffle)
        .batch(batch_size, drop_remainder=drop_remainder).map(tf_getdata, num_parallel_calls=tf.data.AUTOTUNE))

    tfds = tfds.map(tf_split, num_parallel_calls=tf.data.AUTOTUNE)
    
    if lrepeat:
        tfds = tfds.repeat()

    return tfds

class StreamMonthlyNetCDF(object):
    def __init__(self, mode: str, datadir: Path, patt: str, nfiles_merge: Union[int, Dict], predictands: List,
                 predictors: List = None, static_predictors: List = None, sample_dim: str = "time", norm_dims: List = None,
                 norm_obj=None ,with_horovod: bool = False, seed: int = None, nworkers: int = 10):
        """
        Class object providing all methods to create a TF dataset that iterates over a set of (monthly) netCDF-files
        rather than loading all into memory. Instead, only a subset of all netCDF-files is loaded into memory.
        Furthermore, the class attributes provide key information on the handled dataset
        :param datadir: directory where set of netCDF-files are located
        :param patt: filename pattern to allow globbing for netCDF-files
        :param nfiles_merge: number of files per data subset loaded into memory (can be an integer or a dictionary like {"#GPUS=1": 33})
        :param predictands: list of predictand variables names to be obtained
        :param predictors: list of predictor variable names to be obtained, pass None
                           if all vars with suffix _in should be chosen
        :param static_predictors: list of static predictor variable names to be obtained
        :param sample_dim: name of dimension in the data over which sampling should be performed
        :param var_tar2in: predictand (target) variable that can be inputted as well
                          (e.g. static variables known a priori such as the surface topography)
        :param norm_dims: list of dimensions over which data will be normalized
        :param norm_obj: normalization object providing parameters for (de-)normalization
        :param with_horovod: flag to trigger horovod-based distributed dataset creation
        :param seed: seed for random sampling of netCDF-files
        :param nworkers: number of threads to read the netCDF-files
        """
        self.with_horovod = with_horovod
        if self.with_horovod:
            import horovod.tensorflow as hvd
        self.seed = seed
        self.stream_mode = mode
        self.data_dir = datadir
        # get file list and number of files to be merged for data subse
        self.file_list = patt
        self.nfiles = len(self.file_list)
        # get relevant data dimensions
        ds_all = xr.open_mfdataset(list(self.file_list), decode_cf=False, cache=False)  # , parallel=True)
        self.all_dims = ds_all.dims
        self.sample_dim = sample_dim
        self.nsamples = ds_all.dims[sample_dim]
        self.dataset_size = self.get_dataset_size()
        # sampling of datafiles
        self.file_list_random = random.sample(self.file_list, self.nfiles)
        self.nfiles2merge = nfiles_merge                                # number of files to be merged for data subset  
        self.nfiles_merged = int(self.nfiles / self.nfiles2merge)       # number of data subsets
        self.samples_merged = self.get_samples_per_merged_file()
        # list of files can be larger for distributed training, i.e. effective dataset size can be increased
        if self.with_horovod:
            # re-do dataset calculation since file list is potentially larger for distributed training
            self.effective_dataset_size = self.get_dataset_size(random_list=True)
        else:
            self.effective_dataset_size = self.dataset_size
        # handle selected variables
        self.varnames_list = self.get_all_varnames()
        self.predictor_list = predictors
        self.static_predictor_list = static_predictors
        self.predictand_list = predictands
        self.n_predictands, self.n_predictors = len(self.predictand_list), len(self.predictor_list)
        self.all_vars = self.predictor_list + self.predictand_list 
        if self.static_predictor_list is not None:
            self.all_vars = self.static_predictor_list + self.all_vars     # ordering important to ensure that predictors come first (cf. make_tf_dataset_allmem-method)!
            self.n_predictors += len(self.static_predictor_list) 
        self.data_xy_dim = self.get_nxy_dim(ds_all)
        # sanity check on shapes of predictors, predictands and static predictors depending on stream_mode
        self.check_data_shapes()
        # get normalization object
        t0 = timer()
        # check if normalization object is provided
        self.normalization_time = -999.
        if norm_obj is None:
            print("Start computing normalization parameters.")
            self.data_norm = ZScore(norm_dims)  # TO-DO: Allow for arbitrary normalization
            self.norm_params = self.data_norm.get_required_stats(ds_all)
            self.normalization_time = timer() - t0
        else:
            self.data_norm = norm_obj
            self.norm_params = norm_obj.norm_stats

        # initialize data loading
        self.data_loaded = [xr.Dataset, xr.Dataset]        # two datasets will be cached
        self.iload_next, self.iuse_next = 0, 0
        self.reading_times = []
        self.ds_proc_size = 0.
        self.data_now = None
        if not nworkers:
            nworkers = min((multiprocessing.cpu_count(), self.nfiles2merge))
        self.pool = ThreadPool(nworkers)
        
        del ds_all
        gc.collect()

    @property
    def stream_mode(self):
        return self._stream_mode
    
    @stream_mode.setter
    def stream_mode(self, mode):
        known_modes = ["hi_input", "hi_input_named_target", "lo_input"]
        if mode not in known_modes:
            raise ValueError(f"Streaming mode {mode} is not supported. Known modes are {', '.join(known_modes)}")
        
        self._stream_mode = mode
        
    @property
    def data_dir(self):
        return self._data_dir

    @data_dir.setter
    def data_dir(self, datadir):
        if not os.path.isdir(datadir):
            raise NotADirectoryError(f"Parsed data directory '{datadir}' does not exist.")

        self._data_dir = datadir

    @property
    def file_list(self):
        return self._file_list

    @file_list.setter
    def file_list(self, patt):
        patt = patt if patt.endswith(".nc") else f"{patt}.nc"
        files = glob.glob(os.path.join(self.data_dir, patt))

        if not files:
            raise FileNotFoundError(f"Could not find any files with pattern '{patt}' under '{self.data_dir}'.")

        self._file_list = list(
            np.asarray(sorted(files, key=lambda s: int(re.search(r'\d+', os.path.basename(s)).group()))))

    @property
    def seed(self):
        return self._seed 

    @seed.setter 
    def seed(self, seed_int):
        if self.with_horovod and seed_int is None:
            raise ValueError(f"Seed integer must be provided for distitributed training with Horovod,")
        
        # set seed
        random.seed(seed_int)
        self._seed = seed_int

    @property
    def nfiles2merge(self):
        return self._nfiles2merge
    
    @nfiles2merge.setter
    def nfiles2merge(self, n2merge: Union[int, Dict]):

        if isinstance(n2merge, int):
            n = n2merge
        else: 
            n = n2merge[f"#GPUS={hvd.size()}"] if self.with_horovod else n2merge[f"#GPUS=1"]

        # ensure that n is a divisor of the total number of files
        n = find_closest_divisor(self.nfiles, n)

        self._nfiles2merge = n
        # for distributed training, data files must be distributed over workers
        if self.with_horovod:
            if hvd.rank() == 0:
                if n != n2merge:
                    print(f"{n2merge} is not a divisor of the total number of files. Value is changed to {n}")
                print(f"Distributed streaming over {hvd.size()} workers.")

            assert n > hvd.size(), f"Number of files to merge {n} must be larger than number of workers {hvd.size()}."
            self._nfiles2merge = int(n / hvd.size())
            if n % hvd.size() > 0:
                # In case that the modulo is non-zero, nfiles2merge is incremented and the file list is appended
                # so that each work processes the same number of files.
                # Note that duplicated files only occur in the last data subset. 
                # To avoid duplicates in the last subset itself, only files from the preceiding subsets are appended.
                self._nfiles2merge += 1
                nfiles_req = int(hvd.size() * self._nfiles2merge * self.nfiles/n)
                if hvd.rank() == 0: 
                    print(f"Append file list by {nfiles_req - self.nfiles} files to get {nfiles_req} files ({self._nfiles2merge} files per worker).")
                self.file_list_random += random.sample(self.file_list_random[0:self.nfiles-n], nfiles_req - self.nfiles)
                self.nfiles = len(self.file_list_random)
        else:
            if n != n2merge:
                print(f"{n2merge} is not a divisor of the total number of files. Value is changed to {n}")


    @property
    def sample_dim(self):
        return self._sample_dim

    @sample_dim.setter
    def sample_dim(self, sample_dim):
        if not sample_dim in self.all_dims:
            raise KeyError(f"Could not find dimension '{sample_dim}' in data.")

        self._sample_dim = sample_dim

    @property
    def predictor_list(self):
        return self._predictor_list

    @predictor_list.setter
    def predictor_list(self, selected_predictors: List):
        """
        Initalizes predictor list. In case that selected_predictors is set to None, all variables with suffix `_in`
        in their names are selected.
        In case that a list of selected_predictors is parsed, their availability is checked
        :param selected_predictors: list of predictor variables or None
        """
        self._predictor_list = self.check_and_choose_vars(selected_predictors, "_in")
        
    @property
    def static_predictor_list(self):
        return self._static_predictor_list
    
    @static_predictor_list.setter
    def static_predictor_list(self, selected_static_predictors: List):
        if selected_static_predictors is None:
            # if no static, high-res predictors are added, set to None
            self._static_predictor_list = None
        else:
            self._static_predictor_list = self.check_and_choose_vars(selected_static_predictors)

    @property
    def predictand_list(self):
        return self._predictand_list

    @predictand_list.setter
    def predictand_list(self, selected_predictands: List):
        """
        Similar to predictor_list-setter, but does not allow for parsing None.
        """
        assert isinstance(selected_predictands, list), "Selected predictands must be a list of variable names"
        self._predictand_list = self.check_and_choose_vars(selected_predictands, "_tar")

    def __len__(self):
        return self.nsamples

    def getitems(self, indices):
        """
        Return samples from loaded dataset, either as single array (mode: 'hi_input' and 'hi_input_named_target')
        or as tuple of arrays (mode: 'lo_input')
        :param indices: sample indices 
        """
        if self.stream_mode == "lo_input":
            da_now = self.getitems_as_array(indices)
        else:
            da_now = self.getitems_as_tuple(indices)
        
        return da_now
    
    def getitems_as_array(self, indices):
        """
        Retrieves samples from dataset and returns an ordered array for later data handling.
        :param indices: sample indices 
        :return: Ordered array of variables with variables as last dimension. Order is: [static_predictors], predictors, predictands
        """
        da_now = self.data_now.isel({self.sample_dim: indices}).to_array("variables").sel({"variables": self.all_vars})
        
        return da_now.transpose(..., "variables")        
        
    def getitems_as_tuple(self, indices):
        """
        Retrieves samples from dataset and returns an ordered tuple of arrays for later data handling.
        :param indices: sample indices 
        :return: Ordered tuple of arrays with variables as last dimension. Order is: static_predictors, predictors, predictands
        """
        da_in_coa, da_in_static, da_out = self.data_now[self.predictor_list].isel({self.sample_dim: indices}).to_array("variables"), \
                                          self.data_now[self.static_predictor_list].isel({self.sample_dim: indices}).to_array("variables"), \
                                          self.data_now[self.predictand_list].isel({self.sample_dim: indices}).to_array("variables")
        
        da_tuple = (da_in_static.transpose(..., "variables"), da_in_coa.transpose(..., "variables"), da_out.transpose(..., "variables"))
        return da_tuple

    def get_dataset_size(self, random_list: bool = False):
        """
        Sum the size of all dataset files in bytes.
        :param random_list: if True, the size computation is based on randomized list which might be longer for distributed training
        :return: size of dataset files in bytes 
        """
        dataset_size = 0.
        # iterate over file_list_random-attribute since this comprises the actual files that are streamed 
        # incl. duplicates in case of distributed training
        flist = self.file_list_random if random_list else self.file_list 
        for datafile in flist:
            dataset_size += os.path.getsize(datafile)

        return dataset_size

    def get_nxy_dim(self, ds):
        """
        Retrieve the spatial dimensionality of the input and target data.
        :return: Dictionary of spatial dimensions of the predictands, predictors and, if available, static predictors
        """
        data_dims_keys = ["output", "input",]
        infer_vars = [self.predictand_list[0], self.predictor_list[0]]
        
        if self.static_predictor_list is not None:
            data_dims_keys += ["input_static"]
            infer_vars += [self.static_predictor_list[0]]

        dim_dict = {}
        for key, var in zip(data_dims_keys, infer_vars):
            dimnames = list(ds[var].dims)
            dimnames.remove(self.sample_dim)
            
            print(dimnames)
            print(self.all_dims)

            data_dim = itemgetter(*dimnames)(self.all_dims)
            dim_dict[key] = data_dim
            
        return dim_dict

    def get_samples_per_merged_file(self):
        nsamples_merged = []

        for i in range(self.nfiles_merged):
            file_list_now = self.file_list_random[i * self.nfiles2merge: (i + 1) * self.nfiles2merge]
            ds_now = xr.open_mfdataset(list(file_list_now), decode_cf=False)
            nsamples_merged.append(ds_now.dims[self.sample_dim])  

        return max(nsamples_merged)

    def get_all_varnames(self):
        ds_test = xr.open_dataset(self.file_list[0])
        return list(ds_test.variables)

    def check_and_choose_vars(self, var_list: List[str], suffix: str = "*"):
        """
        Checks list of variables for availability or retrieves all variables named with a given suffix
        (for var_list = None)
        :param var_list: list of predictor variables or None
        :param suffix: optional suffix of variables to selected. Only effective if var_list is None
        :return selected_vars: list of selected variables
        """
        if var_list is None:
            selected_vars = [var for var in self.varnames_list if var.endswith(suffix)]
        else:
            stat_list = [var in self.varnames_list for var in var_list]
            if all(stat_list):
                selected_vars = var_list
            else:
                miss_inds = [i for i, x in enumerate(stat_list) if not x]
                miss_vars = [var_list[i] for i in miss_inds]
                raise ValueError(f"Could not find the following variables in the dataset: {*miss_vars,}")

        return selected_vars
    
    def check_data_shapes(self):
        """
        Check if the spatial data dimensions are consistent w.r.t. to the streaming mode.
        """
        nxy_in_str, nxy_stat_str = [str(n) for n in self.data_xy_dim['input']], [str(n) for n in self.data_xy_dim['input_static']]
        nxy_out_str = [str(n) for n in self.data_xy_dim['output']]
        
        if self.stream_mode == "lo_input":
            assert self.data_xy_dim["input"] != self.data_xy_dim["input_static"], f"Predictors and static predictors must have different spatial shapes. " + \
                                                                                      f"predictors: [{','.join(nxy_in_str)}], " + \
                                                                                      f"static_predictors: [{','.join(nxy_stat_str)}]"
            assert self.data_xy_dim["output"] == self.data_xy_dim["input_static"], f"Predictands and static predictors must have the same spatial shapes. " + \
                                                                                      f"predictands: [{','.join(nxy_out_str)}], " + \
                                                                                      f"static_predictors: [{','.join(nxy_stat_str)}]"
        else:
            mess = f"The spatial shapes of all variables must be the same. predictands: [{','.join(nxy_out_str)}], predictors: [{','.join(nxy_in_str)}]"
            if self.static_predictor_list is not None:
                mess += f" static_predictors: [{','.join(nxy_stat_str)}]"
            assert self.data_xy_dim["input"] == self.data_xy_dim["input_static"] == self.data_xy_dim["output"], mess        

    @staticmethod
    def _process_one_netcdf(fname, data_norm, engine: str = "netcdf4", var_list: List = None, **kwargs):
        with xr.open_dataset(fname, decode_cf=False, engine=engine, **kwargs) as ds_now:
            if var_list: ds_now = ds_now[var_list]
            ds_now = StreamMonthlyNetCDF._preprocess_ds(ds_now, data_norm)
            ds_now = ds_now.load()
            return ds_now

    @staticmethod
    def _preprocess_ds(ds, data_norm):
        ds = data_norm.normalize(ds)
        return ds.astype("float32")

    def _read_mfdataset(self, files, **kwargs):
        # parallel processing of files incl. normalization
        datasets = self.pool.map(partial(self._process_one_netcdf, data_norm=self.data_norm, **kwargs), files)
        ds_all = xr.concat(datasets, dim=self.sample_dim)
        # clean-up
        del datasets
        gc.collect()

        return ds_all

    def read_netcdf(self, set_ind):
        set_ind = tf.keras.backend.get_value(set_ind)
        set_ind = int(str(set_ind).lstrip("b'").rstrip("'"))
        set_ind = int(set_ind % self.nfiles_merged)
        file_list_now = self.file_list_random[set_ind * self.nfiles2merge:(set_ind + 1) * self.nfiles2merge]
        il = int(self.iload_next % 2)
        # read the normalized data into memory
        # ds_now = xr.open_mfdataset(list(file_list_now), decode_cf=False, data_vars=self.all_vars,
        #                           preprocess=partial(self._preprocess_ds, data_norm=self.data_norm),
        #                           parallel=True).load()
        t0 = timer()
        # Restriction to read dynamic variables is not required currently,
        # since constant data get automatically broadcasted with the _read_mfdataset-method
        #data_now = self._read_mfdataset(file_list_now, var_list=self.dyn_vars).copy()
        data_now = self._read_mfdataset(file_list_now, var_list=self.all_vars).copy()
        nsamples = data_now.sizes[self.sample_dim]

        if nsamples < self.samples_merged:
            t1 = timer()
            add_samples = self.samples_merged - nsamples
            istart = random.randint(0, self.samples_merged - add_samples - 1)
            # slice data from data_now...
            ds_add = data_now.isel({self.sample_dim: slice(istart, istart+add_samples)})
            if ds_add.sizes[self.sample_dim] != add_samples:
                print("WARNING: ds_add contains inconsistent number of samples. Re-try...")
                add_samples = self.samples_merged - nsamples
                istart = random.randint(0, self.samples_merged - add_samples - 1)
                ds_add = data_now.isel({self.sample_dim: slice(istart, istart + add_samples)})
            # ... and modify underlying sample-dimension to allow clean concatenation
            ds_add[self.sample_dim] = data_now[self.sample_dim][-1].values + 1 + np.arange(add_samples)
            ds_add[self.sample_dim] = ds_add[self.sample_dim].assign_attrs(data_now[self.sample_dim].attrs)
            data_now = xr.concat([data_now, ds_add], dim=self.sample_dim)
            print(f"Appending data with {add_samples:d} samples took {timer() - t1:.2f}s" +
                  f"(total #samples: {data_now.sizes[self.sample_dim]})")
            
        # Appending with constant variables is not required since they are read and broadcast to data_now already (see above)
        #if self.const_vars:
        #    ds_const_append = self.ds_const.copy().expand_dims({self.sample_dim: data_now[self.sample_dim]})
        #    data_now = xr.merge([data_now, ds_const_append])

        # write to class attribute
        self.data_loaded[il] = data_now
        # timing
        t_read = timer() - t0
        self.reading_times.append(t_read)
        self.ds_proc_size += data_now.nbytes
        print(f"Dataset #{set_ind:d} ({il+1:d}/2) reading time: {t_read:.2f}s.")
        self.iload_next = il + 1

        return il

    def choose_data(self, _):
        ik = int(self.iuse_next % 2)
        self.data_now = self.data_loaded[ik]
        print(f"Use data subset {ik:d}...")
        self.iuse_next = ik + 1
        return True

# Modified normalization class
Note that this is mandatory since the data has differing coordinates for target and input data is not yet supported by the normalization class.
To-Do:
- [ ] Revise Normalize-class accordingly
Here, an ad-hoc fix is made to allow passing of `norm_dims=None` which results into averaging over all data dimensions.

In [ ]:
da_or_ds = Union[xr.DataArray, xr.Dataset]

class Normalize(ABC):
    """
    Abstract class for normalizing data.
    """

    def __init__(self, method: str, norm_dims: List):
        self.method = method
        self.norm_dims = norm_dims
        self.norm_stats = None

    def normalize(self, data: xr.DataArray, **stats):
        """
        Normalize data
        :param data: The DataArray to be normalized
        :param stats: Optional parameters to perform normalization. Must fit to normalization type!
        :return: DataArray with normalized data
        """
        # sanity checks
        # if not isinstance(data, xr.DataArray):
        #    raise TypeError(f"Passed data must be a xarray.DataArray, but is of type {str(type(data))}.")

        # do the computation
        norm_stats = self.get_required_stats(data, **stats)
        norm_stats = Normalize.match_datatype(data, *norm_stats)
        data_norm = self.normalize_data(data, *norm_stats)

        return data_norm

    def denormalize(self, data: da_or_ds, **stats):
        """
        Denormalize data.
        :param data: The DataArray to be denormalized.
        :param stats: Optional parameters to perform denormalization. Must fit to normalization type!
        :return: DataArray with denormalized data.
        """
        # sanity checks
        # if not isinstance(data, xr.DataArray):
        #    raise TypeError(f"Passed data must be a xarray.DataArray, but is of type {str(type(data))}.")

        # do the computation
        norm_stats = self.get_required_stats(data, **stats)
        norm_stats = Normalize.match_datatype(data, *norm_stats)
        data_denorm = self.denormalize_data(data, *norm_stats)

        return data_denorm

    @property
    def norm_dims(self):
        return self._norm_dims

    @norm_dims.setter
    def norm_dims(self, norm_dims):
        self._norm_dims = list(norm_dims) if norm_dims is not None else None

    def _check_norm_dims(self, data):
        """
        Check if dimension for normalization reside in dimensions of data.
        :param data: the data (xr.DataArray) to be normalized
        :return True: in case of passed check, a ValueError is risen else
        """
        data_dims = list(data.dims)
        norm_dims_check = [norm_dim in data_dims for norm_dim in self.norm_dims]
        if not all(norm_dims_check):
            imiss = np.where(~np.array(norm_dims_check))[0]
            miss_dims = list(np.array(self.norm_dims)[imiss])
            raise ValueError("The following dimensions do not reside in the data: " +
                             f"{', '.join(miss_dims)}")

        return True

    @staticmethod
    def match_datatype(data, *args, var_dim="variables"):
        """
        Ensures that the arguments have the same xarray datatype (either xr.DataArray or xr.Dataset) as data,
        i.e. coerces all arguments against type(data) if necessary.
        :param data: the reference data (must be either xr.Dataset or xr.DataArray)
        :param args: arbitrary number of arguments (all of them must also be either xr.Dataset or xr.DataArray,
                     but should not be mixed, e.g. type(args[0])=xr.Dataset and type(args[1])=xr.DataArray is not
                     allowed
        :param var_dim: dimension name to convert from/to xr.Dataset/xr.DataArray
        """

        # sanity check
        ds_or_da = (xr.Dataset, xr.DataArray)
        all_args = [data] + list(args)
        if not all(isinstance(arg, ds_or_da) for arg in all_args):
            flags = [not isinstance(arg, ds_or_da) for arg in all_args]
            inds = np.nonzero(flags)[0].tolist()
            if len(inds) == 1:
                err_str = f"The parsed argument at position {inds} is"
            else:
                err_str = f"The parsed arguments at positions {inds} are"
            raise ValueError(f"{err_str} not an xarray.DataArray or xarray.Dataset.")

        # align type of arguments if required
        if isinstance(data, type(args[0])):
            args_new = args
        elif isinstance(data, xr.Dataset) and isinstance(args[0], xr.DataArray):
            args_new = tuple(arg.to_dataset(dim=var_dim) for arg in args)
        elif isinstance(data, xr.DataArray) and isinstance(args[0], xr.Dataset):
            args_new = tuple(arg.to_array(dim=var_dim) for arg in args)
        else:
            raise ValueError("Unknown error occured. Please check all input parameters.")

        return args_new

    def save_norm_to_file(self, js_file, missdir_ok: bool = True):
        """
        Write normalization parameters to file
        :param js_file: Path to JSON-file to be created
        :param missdir_ok: If True, base-directory of JSON-file can be missing and will be created then
        :return: -
        """
        if self.norm_stats is None:
            raise AttributeError("norm_stats is still None. Please run (de-)normalization to get parameters.")

        if any([stat is None for stat in self.norm_stats.values()]):
            raise AttributeError("Some parameters of norm_stats are None.")

        norm_serialized = {key: da.to_dict() for key, da in self.norm_stats.items()}

        # serialization and (later) deserialization depends on data type.
        # Thus, we have to save it to the dictionary
        d0 = list(self.norm_stats.values())[0]
        if isinstance(d0, xr.DataArray):
            norm_serialized["data_type"] = "data_array"
        elif isinstance(d0, xr.Dataset):
            norm_serialized["data_type"] = "data_set"

        if missdir_ok: os.makedirs(os.path.dirname(js_file), exist_ok=True)

        with open(js_file, "w") as jsf:
            js.dump(norm_serialized, jsf)

    def read_norm_from_file(self, js_file):
        """
        Read normalization parameters from file. Inverse function to write_norm_from_file.
        :param js_file: Path to JSON-file to be read.
        :return: Parameters set to self.norm_stats
        """
        with open(js_file, "r") as jsf:
            norm_data = js.load(jsf)

        data_type = norm_data.pop('data_type', None)

        if data_type == "data_array":
            xr_obj = xr.DataArray
        elif data_type == "data_set":
            xr_obj = xr.Dataset
        else:
            raise ValueError(
                f"Unknown data_type {data_type} in {js_file}. Only 'data_array' or 'data_set' are allowed.")

        norm_data.pop('data_type', None)

        norm_dict_restored = {key: xr_obj.from_dict(da_dict) for key, da_dict in norm_data.items()}

        self.norm_stats = norm_dict_restored

    @abstractmethod
    def get_required_stats(self, data, varname, *stats):
        """
        Function to retrieve either normalization parameters from data or from keyword arguments
        """
        pass

    @staticmethod
    @abstractmethod
    def normalize_data(data, *norm_param):
        """
        Function to normalize data.
        """
        pass

    @staticmethod
    @abstractmethod
    def denormalize_data(data, *norm_param):
        """
        Function to denormalize data.
        """
        pass

da_or_ds = Union[xr.DataArray, xr.Dataset]


class ZScore(Normalize):
    def __init__(self, norm_dims: List):
        super().__init__("z_score", norm_dims)
        self.norm_stats = {"mu": None, "sigma": None}

    def get_required_stats(self, data: da_or_ds, varname: str= None, **stats):
        """
        Get required parameters for z-score normalization. They are either computed from the data
        or can be parsed as keyword arguments.
        :param data: the data to be (de-)normalized
        :param varname: retrieve parameters for specific varname only (without effect if parameters must be retrieved from data)
        :param stats: keyword arguments for mean (mu) and standard deviation (std) used for normalization
        :return (mu, sigma): Parameters for normalization
        """
        mu, std = stats.get("mu", self.norm_stats["mu"]), stats.get("sigma", self.norm_stats["sigma"])

        if mu is None or std is None:
            print("Retrieve mu and sigma from data...")
            mu, std = data.mean(self.norm_dims), data.std(self.norm_dims)
            # the following ensure that both parameters are computed in one graph!
            # This significantly reduces memory footprint as we don't end up having data duplicates
            # in memory due to multiple graphs (and also seem to enfore usage of data chunks as well)
            mu, std = dask.compute(mu, std)
            self.norm_stats = {"mu": mu, "sigma": std}
        else:
            if varname:
                if isinstance(mu, xr.DataArray):
                    mu, std = mu.sel({"variables": varname}), std.sel({"variables": varname})
                elif isinstance(mu, xr.Dataset):
                    mu, std = mu[varname], std[varname]
                else:
                    raise ValueError(f"Unexpected data type for mu and std: {type(mu)}, {type(std)}")
        #    print("Mu and sigma are parsed for (de-)normalization.")

        return mu, std

    @staticmethod
    def normalize_data(data, mu, std):
        """
        Perform z-score normalization on data
        :param data: Data array of interest
        :param mu: mean of data for normalization
        :param std: standard deviation of data for normalization
        :return data_norm: normalized data
        """
        data = (data - mu) / std

        return data

    @staticmethod
    def denormalize_data(data, mu, std):
        """
        Perform z-score denormalization on data.
        :param data: Data array of interest
        :param mu: mean of data for denormalization
        :param std: standard deviation of data for denormalization
        :return data_norm: denormalized data
        """
        data = data * std + mu

        return data

## Test make_tf_dataset_allmem data pipeline (streaming from a single netCDF)

### Get the data
Stream data from example data-file.

In [ ]:
# set diretcories and (hyper-)parameters for WGAN
data_dir = Path("/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/coarse_input/val")
t2m_test_file = data_dir.joinpath("downscaling_benchmark_t2m_val.nc")
js_norm = data_dir.parents[0].joinpath("norm.json")
# datadir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5_michael/preprocessed_era5_ifs/netcdf_data/all_files/"
# outdir = "/p/project/deepacf/maelstrom/langguth1/downscaling_jsc_repo/downscaling_unet/trained_models"

lr_gen = 5.0e-05
lr_gen_end = lr_gen / 10.0
lr_critic = 1.0e-06
lr_decay = True
nepochs = 1
d_steps = 5
batch_size_demo = 2

# get normalization instance
data_norm = ZScore(None)
data_norm.read_norm_from_file(js_norm)

# read raw data...
ds_train = xr.open_dataset(t2m_test_file)

#...and normalize
ds_train = data_norm.normalize(ds_train)
ds_val = ds_train
z_branch = False
print("Datasets for trining, validation and testing loaded.")

# wgan_model = HarrisWGAN(GeneratorHarris, DiscriminatorHarris,
#                  {"lr_decay": lr_decay, "lr_gen": lr_gen, "lr_critic": lr_critic, "lr_gen_end": lr_gen_end,
#                   "train_epochs": nepochs, "d_steps": d_steps, "z_branch": z_branch})

Set-up data pipeline (same for training and validation for simplicity)

In [ ]:
tfds = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * (d_steps + 1),
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t_ml115_in"],
    ["fr_land_tar", "hsurf_tar"],
)
tfds_val = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * (d_steps + 1),
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t_ml115_in"],
    ["fr_land_tar", "hsurf_tar"],
)

In [ ]:
tfds.element_spec[0]

### Start training 

In [ ]:
importlib.reload(sys.modules["model_engine"])
importlib.reload(sys.modules["harris_wgan_model"])

# some prerequisites to instantiate the model and run training
shape_in = {
    "harris_generator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "noise_input": (32, 36, 4),
    },
    "harris_discriminator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "output": (128, 144, 1),
    },
}
varnames_tar = "t_2m_tar"
hparams_dict = dict()                    # make use of defaults!
model_savedir = ""
steps_per_epoch = 10

model_instance = ModelEngine("harris_wgan")
# data prep here
model = model_instance(
    shape_in, list(varnames_tar), hparams_dict, model_savedir, "demo"
)
model.compile(**model.compile_options)
history = model.fit(
    x=tfds,
    epochs=model.hparams["nepochs"],
    steps_per_epoch=steps_per_epoch,
    validation_data=tfds_val,
    validation_steps=steps_per_epoch,
    verbose=1,
    **model.fit_options
)

In [ ]:
model_name = "harriswgan_lr1e-05_epochs1_opt_split_era5_ifs"

savedir = os.path.join("../downscaling_harriswgan/trained_models/", model_name)
os.makedirs(savedir, exist_ok=True)

In [ ]:
model.generator.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_generator")
)
model.critic.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_discriminator")
)

## Test make_tf_dataset_dyn data pipeline (streaming from a set of netCDF-file)

In [ ]:
# paramters
datadir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/coarse_input/val"
file_patt = "downscaling_benchmark_t2m_train_*.nc"
stream_mode = "lo_input"
nfiles_load = 4
norm_dims = ["rlat_in", "rlon_in", "rlat_tar", "rlon_tar"]
nworkers = 4
batch_size = 2
nepochs = 1
    
predictands = ["t_2m_tar"]
predictors= ["t2m_in", "slhf_in", "sshf_in", "sp_in", "z_in", "lsm_in", "t_ml115_in", "t_ml122_in", "t_ml127_in", "t_ml131_in", "t_ml135_in"]
static_predictors= ["hsurf_tar", "fr_land_tar"]


### Get the data
Stream data from example data-file.

In [ ]:
ds_obj = StreamMonthlyNetCDF(stream_mode, datadir, file_patt, nfiles_load, predictands, predictors, static_predictors, 
                             norm_dims = norm_dims, nworkers=nworkers)

tfds=make_tf_dataset_dyn(ds_obj, batch_size=batch_size, nepochs=nepochs, nshuffle = 1000, lrepeat= True, drop_remainder = True)

In [ ]:
tfds.element_spec[0]

In [ ]:
ds_obj.data_xy_dim["input_static"] = (128, 144)
  
print(ds_obj.data_xy_dim["input"])
print(ds_obj.data_xy_dim["input_static"])
print(ds_obj.data_xy_dim["output"])

if ds_obj.stream_mode == "lo_input":
    nxy_in_str, nxy_stat_str, nxy_out_str = [str(n) for n in ds_obj.data_xy_dim['input']], [str(n) for n in ds_obj.data_xy_dim['input_static']], [str(n) for n in ds_obj.data_xy_dim['output']]
    assert ds_obj.data_xy_dim["input"] != ds_obj.data_xy_dim["input_static"], f"Predictors and static predictors must have different spatial shapes. " + \
                                                                              f"predictors: [{','.join(nxy_in_str)}], " + \
                                                                              f"static_predictors: [{','.join(nxy_stat_str)}]"
    assert ds_obj.data_xy_dim["output"] == ds_obj.data_xy_dim["input_static"], f"Predictands and static predictors must have the same spatial shapes. " + \
                                                                              f"predictands: [{','.join(nxy_out_str)}], " + \
                                                                              f"static_predictors: [{','.join(nxy_stat_str)}]"
else:
    assert ds_obj.data_xy_dim["input"] == ds_obj.data_xy_dim["input_static"] == ds_obj.data_xy_dim["output"]

In [ ]:
if ds_obj.stream_mode == "lo_input":
    shape_in = [*ds_obj.data_xy_dim["input"], len(ds_obj.predictor_list), len(ds_obj.static_predictor_list)]
else:
    shape_in = [*ds_obj.data_xy_dim["input"], len(ds_obj.predictor_list + ds_obj.static_predictor_list)]

In [ ]:
tuple(shape_in)

In [ ]:
dimnames = ['rlat_tar', 'rlon_tar']
all_dims = ['time', 'rlon_in', 'rlat_in', 'rlon_tar', 'rlat_tar']

data_dim = itemgetter(*dimnames)(all_dims)

In [ ]:
ds_all.dims[sample_dim]